# 🏥 ViBioMIR 2026: All-In-One Fast Inference Pipeline
## Multilingual Medical Document Retrieval (Macro F2 Optimized)

Notebook này là giải pháp **All-In-One hoàn chỉnh** tích hợp đầy đủ:
1. **HyDE LLM Queries** (`queries_hyde.jsonl`): Câu hỏi được mở rộng thuật ngữ chuyên khoa & bệnh học lâm sàng từ LLM.
2. **Doc-Index (878.943 bài viết)**: Đảm bảo phủ trọn vẹn **300 Documents** (khai thác tối đa điểm F2 Recall).
3. **Chunk Hybrid Index (FAISS SQ8 + BM25s)**: Đoạn trích 750 ký tự chuẩn chỉ số LCS.
4. **Cross-Encoder Reranker (`bge-reranker-v2-m3`)**: Chạy song song cả 2 GPU Tesla T4 để chọn ra **150 Chunks** chất lượng cao nhất.

> ⚙️ **Thiết lập Kaggle quan trọng trước khi bấm Run All:**
> - **Accelerator**: Chọn **GPU T4 x 2** (Dual GPU).
> - **Internet**: Bật **ON**.
> - **Secrets**: Vào menu **Add-ons -> Secrets**, thêm secret tên `HF_TOKEN` và **tích chọn checkbox** bên cạnh.

In [ ]:
# 1. Kiểm tra phần cứng 2 GPU Tesla T4
!nvidia-smi

In [ ]:
# 2. Tự động Clone hoặc Pull cập nhật Repository từ GitHub
import os
from pathlib import Path

REPO_URL = "https://github.com/neiths/med-doc-retrieval.git"
PROJECT_DIR = Path("/kaggle/working/med-doc-retrieval")

if not PROJECT_DIR.exists():
    print(f"📥 Đang Clone repository từ {REPO_URL}...")
    !git clone -b main {REPO_URL} {PROJECT_DIR}
else:
    print(f"🔄 Đang cập nhật code mới nhất từ GitHub branch main...")
    %cd {PROJECT_DIR}
    !git config --global --add safe.directory {PROJECT_DIR}
    !git pull origin main

%cd {PROJECT_DIR}
print("✓ Thư mục làm việc hiện tại:", os.getcwd())

In [ ]:
# 3. Cài đặt các thư viện cần thiết bằng uv (siêu tốc ~15s)
!pip install -q uv
!uv pip install --system -q \
    "faiss-gpu>=1.7.2" \
    "sentence-transformers>=3.0.0" \
    "FlagEmbedding>=1.2.10" \
    "bm25s[full]>=0.1.10" \
    "pyvi>=0.1.1" \
    "jieba>=0.42.1" \
    "loguru>=0.7.2" \
    "pydantic>=2.7.0"
print("✓ Cài đặt thư viện hoàn tất!")

In [ ]:
# 4. Xác thực Hugging Face Token từ Kaggle Secrets
import os
from huggingface_hub import HfApi, login

HF_TOKEN = os.environ.get("HF_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN or not HF_TOKEN.strip():
    print("⚠️ Lưu ý: Hãy vào menu Add-ons -> Secrets trên Kaggle và tích chọn HF_TOKEN!")
else:
    login(token=HF_TOKEN)
    print("✓ Đã xác thực HF_TOKEN thành công!")

In [ ]:
# 5. Tải đầy đủ bộ chỉ mục từ Hugging Face Bucket (nieths/ViBioMIR)
import glob
from pathlib import Path

os.makedirs("data/indices/bm25s_index", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("outputs/submissions", exist_ok=True)

has_k_dense = bool(glob.glob("/kaggle/input/**/*.faiss", recursive=True))
has_k_sqlite = bool(glob.glob("/kaggle/input/**/chunks_meta*.sqlite", recursive=True))

needed_files = [
    # BM25s Sparse Index
    ("indices/bm25s_index/data.csc.index.npy", "data/indices/bm25s_index/data.csc.index.npy"),
    ("indices/bm25s_index/indices.csc.index.npy", "data/indices/bm25s_index/indices.csc.index.npy"),
    ("indices/bm25s_index/indptr.csc.index.npy", "data/indices/bm25s_index/indptr.csc.index.npy"),
    ("indices/bm25s_index/params.index.json", "data/indices/bm25s_index/params.index.json"),
    ("indices/bm25s_index/vocab.index.json", "data/indices/bm25s_index/vocab.index.json"),
    ("indices/bm25s_metadata.json", "data/indices/bm25s_metadata.json"),
    # PubMed cache & Enriched queries
    ("processed/pubmed_cache.jsonl", "data/processed/pubmed_cache.jsonl"),
    ("processed/queries_enriched.jsonl", "data/processed/queries_enriched.jsonl"),
    # HyDE queries & Doc-level Index
    ("processed/queries_hyde.jsonl", "data/processed/queries_hyde.jsonl"),
    ("indices/doc_index_sq8.faiss", "data/indices/doc_index_sq8.faiss"),
    ("indices/doc_metadata.parquet", "data/indices/doc_metadata.parquet"),
]

if not has_k_dense:
    needed_files.append(("indices/dense_index_sq8.faiss", "data/indices/dense_index_sq8.faiss"))
    needed_files.append(("indices/dense_metadata_sq8.json", "data/indices/dense_metadata_sq8.json"))
if not has_k_sqlite:
    needed_files.append(("indices/chunks_meta_slim.sqlite", "data/indices/chunks_meta_slim.sqlite"))

api = HfApi(token=HF_TOKEN or None)
missing_files = [(r, l) for r, l in needed_files if not Path(l).exists() or Path(l).stat().st_size == 0]

if missing_files:
    print(f"📥 Đang tải {len(missing_files)} file chỉ mục từ HF Bucket 'nieths/ViBioMIR'...")
    api.download_bucket_files(
        bucket_id="nieths/ViBioMIR",
        files=missing_files,
        token=HF_TOKEN or None,
    )
    print("✅ Đã tải xong toàn bộ các file chỉ mục!")
else:
    print("✅ Tất cả các file chỉ mục đã có sẵn trên máy!")

In [ ]:
# 6. Kiểm tra các file trong data/indices
!ls -lh data/indices

In [ ]:
# 7. Chạy Inference Runner (Tích hợp HyDE + Doc-Index + Chunk Hybrid + Reranker)
import sys
import runpy
import torch
from pathlib import Path

dense_arg = "data/indices"
meta_arg = "data/indices/chunks_meta_slim.sqlite"

k_faiss = glob.glob("/kaggle/input/**/*.faiss", recursive=True)
k_sqlite = glob.glob("/kaggle/input/**/chunks_meta*.sqlite", recursive=True)
if k_faiss:
    dense_arg = str(Path(k_faiss[0]).parent)
if k_sqlite:
    meta_arg = k_sqlite[0]

# Ưu tiên HyDE queries
queries_path = Path("data/processed/queries_hyde.jsonl")
if not queries_path.exists():
    queries_path = Path("data/processed/queries_enriched.jsonl")

print(f"✓ File Queries: {queries_path}")
print(f"✓ Thiết bị GPU: {torch.cuda.device_count()} GPUs -> {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")

ckpt_path = "outputs/submissions/run_v4_hyde_doc.partial.jsonl"

# Cấu hình tham số tối thượng
sys.argv = [
    "scripts/run_inference_only.py",
    "--queries", str(queries_path),
    "--dense-dir", dense_arg,
    "--chunks-meta-path", meta_arg,
    "--doc-index-path", "data/indices/doc_index_sq8.faiss",
    "--doc-metadata-path", "data/indices/doc_metadata.parquet",
    "--hybrid-top-k", "500",          # 500 ứng viên từ cả Dense SQ8 và BM25s
    "--top-k-docs", "300",            # Đảm bảo 300 Docs cho F2 Recall
    "--top-k-chunks", "150",          # 150 Chunks chất lượng cao
    "--max-chunks-per-doc", "3",      # Tối đa 3 chunks/doc
    "--expand-top-docs", "40",        # Mở rộng rerank cho 40 tài liệu hàng đầu
    "--expand-max-chunks-per-doc", "5",
    "--batch-size", "32",             # Chia đều song song trên GPU T4 kép
    "--disable-pubmed-network",       # Chạy 100% offline cache
    "--checkpoint", ckpt_path,
    "--output-name", "submission.json",
]

print("\n🚀 BẮT ĐẦU CHẠY SUY LUẬN TOÀN DIỆN (Dự kiến ~45-55 phút)...\n")
runpy.run_path("scripts/run_inference_only.py", run_name="__main__")

In [ ]:
# 8. Kiểm định file submission và tạo nút tải về máy
import base64
import json
import zipfile
from IPython.display import HTML, display

zip_path = Path("/kaggle/working/submission.zip")
if not zip_path.exists():
    zip_path = Path("outputs/submissions/submission.zip")

if zip_path.exists():
    print(f"🎉 FILE SUBMISSION ĐÃ SẴN SÀNG: {zip_path} ({zip_path.stat().st_size / (1024*1024):.2f} MB)")
    
    # Kiểm tra nhanh số lượng queries và docs
    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open("submission.json") as f:
            sub_data = json.load(f)
    
    d_lens = [len(q["relevant_docs"]) for q in sub_data]
    c_lens = [len(q["relevant_chunks"]) for q in sub_data]
    print(f"• Tổng số queries: {len(sub_data)}")
    print(f"• Docs trung bình/câu: {sum(d_lens)/len(d_lens):.1f} (min={min(d_lens)}, max={max(d_lens)})")
    print(f"• Chunks trung bình/câu: {sum(c_lens)/len(c_lens):.1f} (min={min(c_lens)}, max={max(c_lens)})")
    
    with open(zip_path, "rb") as f:
        data_b64 = base64.b64encode(f.read()).decode()

    html_button = f'''
    <div style="margin: 20px 0;">
        <a href="data:application/zip;base64,{data_b64}" download="submission.zip" 
           style="display:inline-block; padding:14px 28px; background:#22c55e; color:white; 
                  text-decoration:none; border-radius:8px; font-weight:bold; font-size:16px;
                  box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
           ⬇ BẤM VÀO ĐÂY ĐỂ TẢI SUBMISSION.ZIP VỀ MÁY
        </a>
    </div>
    '''
    display(HTML(html_button))
else:
    print("❌ Không tìm thấy file submission.zip! Hãy kiểm tra lại log bước 7.")